# CFD-Drag-2D: float vs double solver storage (M10)

Builds the CPU and CUDA solvers, runs the fast test suite (including the GPU tests for free-slip
walls and float storage), then compares float and double storage on the first dataset cases
(Cd / Cl / status), times both, and runs the Schafer-Turek cylinder in both precisions.
Output: `/kaggle/working/m10_precision/summary.json` (case folders are deleted).

In [ ]:
REPO = "https://github.com/PradeepkrishnaGM/CFD-Drag-2D.git"
COMMIT = "main"   # branch or commit SHA
WORK = "/kaggle/working"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /tmp/repo && git clone -q $REPO /tmp/repo && git -C /tmp/repo checkout -q $COMMIT && git -C /tmp/repo log --oneline -1

In [ ]:
%cd /tmp/repo
!pip install -q -e . --no-deps && pip install -q pytest
import os, subprocess
nvcc_dir = subprocess.run(["bash", "scripts/kaggle_setup_cuda.sh"], capture_output=True, text=True, check=True).stdout.strip().splitlines()[-1]
os.environ["PATH"] = nvcc_dir + ":" + os.environ["PATH"]
!make -s solver && make -s cuda && ls -la build

In [ ]:
!python -m pytest -q -m "not slow" -rs tests/test_cuda.py tests/test_solver.py tests/test_solver_options.py 2>&1 | tail -15

In [ ]:
!python scripts/compare_precision.py --backend cuda --n 128 --bench 512 --out $WORK/m10_precision

In [ ]:
!find $WORK/m10_precision -mindepth 1 -maxdepth 1 -type d -exec rm -rf {} + ; git rev-parse HEAD > $WORK/m10_precision/commit.txt; ls $WORK/m10_precision